In [0]:
# ============================================================
# CAPA GOLD: Métricas de negocio - VERSIÓN FINAL PARA COMMIT
# ============================================================
from pyspark.sql.functions import col, to_date, avg, max, min, count, round

print("📥 Leyendo silver_telemetry...")
df_silver = spark.read.table("silver_telemetry")

df_with_date = df_silver.withColumn("report_date", to_date(col("timestamp")))

print("⚙ Calculando resumen diario por dispositivo...")
df_gold = df_with_date.groupBy("device_id", "report_date").agg(
    count("event_id").alias("total_events"),
    round(avg("speed_kmh"), 2).alias("avg_speed_kmh"),
    round(avg("battery_pct"), 2).alias("avg_battery_pct"), # <- nueva, clave para flota
    max("engine_temp_c").alias("max_engine_temp_c"),
    min("engine_temp_c").alias("min_engine_temp_c")
)

df_gold = df_gold.orderBy(col("report_date").desc(), col("device_id"))

print(f"✅ Filas Gold generadas: {df_gold.count()}")
display(df_gold.limit(10))

# Guardar
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_device_daily_summary")

print("\n✅ ¡LAKEHOUSE COMPLETO!")
print("   🥉 Bronze: raw_telemetry_dirty.csv -> bronze_telemetry")
print("   🥈 Silver: Limpio + dedup -> silver_telemetry")
print("   🥇 Gold: Agregado diario -> gold_device_daily_summary")